# Deep Dives

The deep dives are nine standalone notebooks that extend specific topics from the [Crash Course](/notebooks/prep/index.html). Each is self-contained — you can read them in any order once you have completed the parent notebook it references. They are not required to reach the capstone, but they build the depth expected at senior and staff level in system design interviews, technical reviews, and job post qualifications for roles requiring hands-on AI infrastructure expertise.

The main path deliberately defers certain topics to avoid overwhelming the crash course with breadth at the expense of depth. The deep dives are where those topics get the full treatment: rigorous theory, working implementations, and discussion of production failure modes that a principal engineer is expected to anticipate and mitigate.

## Notebooks

| # | Title | Parent | Est. Time | Job Post Relevance |
|---|---|---|---|---|
| DD:01 | [Advanced RAG](/notebooks/prep/deep-dives/01-advanced-rag.html) | [07](/notebooks/prep/07-rag-pipeline.html) | 3–4 hr | "evaluate and optimize RAG" |
| DD:02 | [Automated Eval Pipelines](/notebooks/prep/deep-dives/02-eval-pipelines.html) | [04](/notebooks/prep/04-eval-concepts.html) | 3–4 hr | "automated testing pipelines" |
| DD:03 | [Agent Memory](/notebooks/prep/deep-dives/03-agent-memory.html) | [08](/notebooks/prep/08-agents-langgraph.html) | 3–4 hr | "Agentic AI" depth |
| DD:04 | [Resilience Patterns](/notebooks/prep/deep-dives/04-resilience.html) | [10](/notebooks/prep/10-model-serving.html), [12](/notebooks/prep/12-ai-infra.html) | 4–5 hr | "circuit breakers, 99.99% uptime" |
| DD:05 | [MLOps: CI/CD for AI](/notebooks/prep/deep-dives/05-mlops.html) | [12](/notebooks/prep/12-ai-infra.html) | 4–5 hr | "CI/CD pipelines", "model registry" |
| DD:06 | [AI Security & Compliance](/notebooks/prep/deep-dives/06-ai-security.html) | [02](/notebooks/prep/02-prompt-engg.html) | 3–4 hr | "secure SDLC", "compliance" |
| DD:07 | [Advanced Embeddings](/notebooks/prep/deep-dives/07-embeddings-advanced.html) | [03](/notebooks/prep/03-rag-concepts.html), [07](/notebooks/prep/07-rag-pipeline.html) | 3–4 hr | "embeddings, vector databases" |
| DD:08 | [Multimodal LLMs](/notebooks/prep/deep-dives/08-multimodal.html) | [01](/notebooks/prep/01-llm-api.html) | 3–4 hr | "innovation & research" |
| DD:09 | [IaC & Container Orchestration](/notebooks/prep/deep-dives/09-iac-containers.html) | [12](/notebooks/prep/12-ai-infra.html) | 4–5 hr | "Terraform", "Kubernetes", "container orchestration" |

## Dependency Map

Each deep dive branches off a specific point in the main path. The map below shows which main notebook to complete before starting each deep dive:

```
Main Path                         Deep Dives
─────────────────────────────     ──────────────────────────────────────
01  LLM APIs                  ──► DD:08  Multimodal LLMs
02  Prompt Engineering        ──► DD:06  AI Security & Compliance
03  RAG Concepts              ──┐
07  Production RAG            ──┼► DD:01  Advanced RAG
                                └► DD:07  Advanced Embeddings
04  LLM Evaluation            ──► DD:02  Automated Eval Pipelines
08  LangGraph Agents          ──► DD:03  Agent Memory
10  Model Serving             ──┐
12  AI Infrastructure         ──┼► DD:04  Resilience Patterns
                                ├► DD:05  MLOps: CI/CD for AI
                                └► DD:09  IaC & Container Orchestration
```

DD:04, DD:05, and DD:09 all require completing notebook 12 first, as they build directly on the containerization and IaC foundations established there.

## What Each Deep Dive Covers

### DD:01 — Advanced RAG

The production RAG notebook (07) covers the standard retrieve-then-generate pipeline with hybrid retrieval. This deep dive goes further into techniques that close the remaining quality gap: **query rewriting** (transforming a user's question into a form that retrieves better), **HyDE** (Hypothetical Document Embeddings — generating a hypothetical answer to improve retrieval), **cross-encoder reranking** (a second-stage model that scores retrieved passages more accurately than approximate nearest-neighbor search), and **multi-vector indexing** (representing documents at multiple granularities simultaneously). All techniques are benchmarked against the baseline from notebook 07 on the SEC filings corpus.

### DD:02 — Automated Eval Pipelines

Notebook 04 introduces the vocabulary and metrics of LLM evaluation. This deep dive operationalizes it: building an **eval harness** that runs automatically in CI/CD on every prompt change, **dataset curation** strategies for collecting representative test cases from production traffic, **LLM-as-judge** patterns for scalable evaluation without human annotation, and **prompt regression detection** — alerting when a prompt change degrades a previously passing metric. The output is a GitHub Actions workflow that gates deployment on eval pass rate.

### DD:03 — Agent Memory

LangGraph agents (08) use in-context state. This deep dive covers the full memory taxonomy: **short-term** (conversation buffer, summarization), **long-term** (vector-store-backed episodic memory across sessions), and **semantic memory** (distilled facts extracted from past interactions). Covers memory compression (to avoid context overflow), retrieval scoring (recency × importance × relevance from the Generative Agents paper), and the engineering tradeoffs between memory backends (Redis, pgvector, in-process).

### DD:04 — Resilience Patterns

Production AI services have a failure surface that pure software services do not: model timeouts, rate limit bursts, context window overflows, and degraded output quality under load. This deep dive implements the full resilience toolkit: **circuit breakers** (Tenacity + custom state machine), **exponential backoff with jitter**, **fallback chains** (GPT-4o → GPT-4o-mini → rule-based), **bulkheads** (isolating failure domains across tenants), and **graceful degradation** (returning a partial result rather than an error). All patterns are demonstrated in the context of the compliance document reviewer service.

### DD:05 — MLOps: CI/CD for AI

Notebook 12 establishes the deployment infrastructure. This deep dive extends it into the full MLOps loop: **model registry** (versioned model artifacts with metadata in MLflow), **drift detection** (monitoring embedding distribution shift and output quality degradation over time), **automated retraining pipelines** (triggered by drift alerts), **shadow deployment** (running a new model version in parallel before promoting it), and **canary releases** (gradually routing traffic to a new version with automatic rollback on metric degradation). Uses AWS S3 + ECR + CloudWatch as the backend.

### DD:06 — AI Security & Compliance

Notebook 02 previews prompt injection as a threat. This deep dive covers the full OWASP LLM Top 10: **indirect prompt injection via retrieved documents** (the attack surface that RAG systems open), **PII detection and redaction** using Microsoft Presidio, **output filtering** with Guardrails AI (schema enforcement, topic avoidance, toxicity), **audit trail design** (immutable logs of every LLM call including input hash, model version, and output hash), and **secrets management** in multi-tenant AI services. Financial services context throughout: GDPR article 22, FINRA recordkeeping obligations, model explainability requirements.

### DD:07 — Advanced Embeddings

Notebooks 03 and 07 use `text-embedding-3-small` as a black box. This deep dive opens it: **embedding model selection** (MTEB benchmark, task-specific model families), **fine-tuning embeddings** for domain adaptation on financial text (contrastive learning with hard negatives), **Matryoshka Representation Learning** (embeddings that can be truncated without catastrophic quality loss), **dimensionality reduction** (PCA + UMAP for visualization and compression), and **ColBERT late interaction** (token-level matching for higher retrieval precision). All evaluated on the SEC filings corpus.

### DD:08 — Multimodal LLMs

The crash course focuses on text. This deep dive covers the vision API: **document understanding** (PDFs with mixed text and charts), **financial chart Q&A** (feeding earnings charts to GPT-4o and extracting structured data), **table extraction from scanned documents**, and **multimodal RAG** (indexing documents at the page level with both text chunks and image embeddings). The running example is an analyst assistant that processes a quarterly earnings presentation end-to-end.

### DD:09 — IaC & Container Orchestration

Notebook 12 covers ECS Fargate deployment with basic Terraform and K8s manifests. This deep dive goes to senior-engineer depth: **Terraform modules and workspaces** (reusable IaC patterns for multi-environment AI platforms), **EKS cluster provisioning with GPU node groups** (spot-backed g4dn.xlarge with taints and tolerations), **KEDA event-driven autoscaling** (scaling on SQS queue depth rather than CPU), **advanced Helm chart authoring** (dependencies, hooks, library charts, helm test), **GitOps with ArgoCD** (pull-based continuous delivery with drift reconciliation), **namespace isolation and resource quotas** for multi-tenant AI platforms, and **cost governance** (Karpenter node provisioning, spot interruption handling, Kubecost allocation). Closes with a 20-item production readiness checklist, plus two coverage-gap sections: **S3 as an AI data store** (bucket layout, versioning, lifecycle rules, presigned URLs) and **OpenTelemetry and Prometheus** (distributed tracing for FastAPI, Prometheus Golden Signals, Grafana PromQL, alerting rules).

## How to Approach Deep Dives

**Priority if you are preparing for a senior fintech AI engineering role:** DD:06 (security/compliance), DD:04 (resilience), and DD:09 (IaC/containers) are the most directly tested in technical design interviews at regulated financial institutions. DD:02 (eval pipelines) is the most commonly underestimated skill gap — most engineers know how to build LLM systems but cannot speak to how they are validated systematically.

**Priority if you are building a portfolio:** DD:01 (advanced RAG) and DD:05 (MLOps) produce the most concrete, demonstrable artifacts — a benchmarked retrieval system and an end-to-end deployment pipeline respectively.

**If you have limited time:** Read the opening section of each deep dive to understand the problem framing, then return to implement the full notebook once you have a specific interview or project that warrants it.

:::{.callout-note}
Deep dives are intentionally more demanding than the main path notebooks. They assume you have run the parent notebook and are comfortable with its code. If a concept in a deep dive is unclear, the parent notebook is the right place to look first.
:::